In [ ]:
#| default_exp vision_patch

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
import torchio as tio
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass, field
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from fastai.data.all import *
from fastai.learner import Learner
from fastMONAI.vision_core import MedImage, MedMask, med_img_reader
from fastMONAI.vision_plot import find_max_slice
from fastMONAI.vision_inference import _do_resize
from fastMONAI.dataset_info import MedDataset, suggest_patch_size
from fastMONAI.vision_augmentation import transforms_to_specs, transforms_from_specs
from fastMONAI.utils import unwrap_compiled_model

In [ ]:
#| export
def _get_default_device() -> torch.device:
    """Get the default device (CUDA if available, else CPU)."""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _warn_config_override(param_name: str, config_value, explicit_value) -> None:
    """Warn when an explicit argument overrides a (differing) config value."""
    if explicit_value is not None and config_value is not None:
        if explicit_value != config_value:
            warnings.warn(
                f"{param_name} mismatch: explicit={explicit_value}, config={config_value}. "
                f"Using explicit argument."
            )


def _extract_tio_transform(tfm):
    """Return the underlying TorchIO transform from a fastMONAI wrapper (via .tio_transform), else the transform unchanged."""
    return getattr(tfm, 'tio_transform', tfm)


def normalize_patch_transforms(tfms: list) -> list:
    """Extract raw TorchIO transforms from a list of fastMONAI wrappers (raw TorchIO transforms pass through).

    Lets the same transform syntax work in both standard and patch-based workflows.

    Args:
        tfms: List of fastMONAI wrappers or raw TorchIO transforms.

    Returns:
        List of raw TorchIO transforms suitable for tio.Compose().

    Example:
        >>> patch_tfms = normalize_patch_transforms([RandomAffine(degrees=10), RandomGamma(p=0.5)])
    """
    if tfms is None:
        return None
    return [_extract_tio_transform(t) for t in tfms]

In [ ]:
# Test _extract_tio_transform and normalize_patch_transforms
from fastMONAI.vision_augmentation import RandomAffine, RandomGamma, RandomFlip, RandomNoise

wrapped_affine = RandomAffine(degrees=10)
extracted = _extract_tio_transform(wrapped_affine)
test_eq(type(extracted), tio.RandomAffine)

wrapped_gamma = RandomGamma(p=0.5)
extracted = _extract_tio_transform(wrapped_gamma)
test_eq(type(extracted), tio.RandomGamma)

raw_affine = tio.RandomAffine(degrees=10)
extracted = _extract_tio_transform(raw_affine)
test_eq(extracted, raw_affine)  # Should be the exact same object

raw_flip = tio.RandomFlip(p=0.5)
extracted = _extract_tio_transform(raw_flip)
test_eq(extracted, raw_flip)

tfms = [RandomAffine(degrees=5), RandomGamma(p=0.5), RandomNoise(p=0.3)]
normalized = normalize_patch_transforms(tfms)
test_eq(len(normalized), 3)
test_eq(type(normalized[0]), tio.RandomAffine)
test_eq(type(normalized[1]), tio.RandomGamma)
test_eq(type(normalized[2]), tio.RandomNoise)

# Mixed list (wrappers + raw TorchIO)
mixed_tfms = [RandomAffine(degrees=5), tio.RandomGamma(p=0.5)]
normalized = normalize_patch_transforms(mixed_tfms)
test_eq(len(normalized), 2)
test_eq(type(normalized[0]), tio.RandomAffine)
test_eq(type(normalized[1]), tio.RandomGamma)

test_eq(normalize_patch_transforms(None), None)

# Patch-based training

> Patch-based training and inference for 3D medical image segmentation using TorchIO's Queue mechanism.

## Configuration

In [ ]:
#| export
_UNET_DIVISOR = 16  # U-Net-style encoders require patch dims divisible by 2^4


@dataclass
class PatchConfig:
    """Configuration for patch-based training and inference.
    
    Args:
        patch_size: Size of patches [x, y, z].
        patch_overlap: Overlap for inference GridSampler (int, float 0-1, or list).
            - Float 0-1: fraction of patch_size (e.g., 0.5 = 50% overlap)
            - Int >= 1: pixel overlap (e.g., 48 = 48 pixel overlap)
            - List: per-dimension overlap in pixels
        samples_per_volume: Number of patches to extract per volume during training.
        sampler_type: Type of sampler ('uniform', 'label', 'weighted').
        label_probabilities: For LabelSampler, dict mapping label values to probabilities.
        queue_length: Maximum number of patches to store in queue.
        queue_num_workers: Number of workers for parallel patch extraction.
        aggregation_mode: For inference, how to combine overlapping patches ('crop', 'average', 'hann').
        apply_reorder: Whether to reorder to RAS+ canonical orientation. Must match between
            training and inference. Defaults to True (the common case).
        target_spacing: Target voxel spacing [x, y, z] for resampling. Must match between
            training and inference.
        preprocessed: If True, data has been preprocessed externally (e.g., via
            preprocess_dataset()). Training will skip reorder, resample, AND
            pre_patch_tfms (e.g., normalization) since they were already applied.
            Inference is unaffected and always applies pre_inference_tfms to raw
            images. Defaults to False.
        padding_mode: Padding mode for CropOrPad when image < patch_size. Default is 0 (zero padding).
          Can be int, float, or string (e.g., 'minimum', 'mean').
        keep_largest_component: If True, keep only the largest connected component
            in binary segmentation predictions. Only applies during inference when
            return_probabilities=False. Defaults to False. Binary-only: for multi-class
            output (more than 2 classes) it is skipped with a warning, since it would
            otherwise merge all classes into one blob and collapse the labels.
        binary_threshold: Decision boundary for single-channel (sigmoid) masks; a voxel is
            foreground when probability >= binary_threshold (matches MONAI AsDiscrete). Only
            applies when return_probabilities=False and n_classes == 1. Defaults to 0.5.
        normalization: Single source of truth for pre-patch / pre-inference intensity
            normalization. A list of fastMONAI transforms (e.g.
            [ZNormalization(masking_method='foreground')]) or JSON spec dicts; coerced to specs
            and persisted with the config. Read by both MedPatchDataLoaders.from_df (training)
            and PatchInferenceEngine (inference). The manual pre_patch_tfms / pre_inference_tfms
            args override this when provided.
    
    Example:
        >>> config = PatchConfig(
        ...     patch_size=[96, 96, 96],
        ...     samples_per_volume=16,
        ...     sampler_type='label',
        ...     label_probabilities={0: 0.1, 1: 0.9},
        ...     target_spacing=[0.5, 0.5, 0.5]
        ... )
    """
    patch_size: list = field(default_factory=lambda: [96, 96, 96])
    patch_overlap: int | float | list = 0
    samples_per_volume: int = 8
    sampler_type: str = 'uniform'
    label_probabilities: dict = None
    queue_length: int = 300
    queue_num_workers: int = 4
    aggregation_mode: str = 'hann'
    # Preprocessing parameters - must match between training and inference
    apply_reorder: bool = True
    target_spacing: list = None
    preprocessed: bool = False  # True = data already preprocessed, skip all preprocessing during training
    padding_mode: int | float | str = 0
    # Post-processing (binary segmentation only)
    keep_largest_component: bool = False
    binary_threshold: float = 0.5  # decision boundary for 1-channel sigmoid masks (>=)
    # Normalization: single source of truth for pre-patch / pre-inference intensity
    # normalization. Accepts live fastMONAI transforms or JSON spec dicts; coerced to
    # specs in __post_init__ so the config stays JSON-serializable.
    normalization: list = None
    
    def __post_init__(self):
        """Validate configuration."""
        valid_samplers = ['uniform', 'label', 'weighted']
        if self.sampler_type not in valid_samplers:
            raise ValueError(f"sampler_type must be one of {valid_samplers}")
        
        valid_aggregation = ['crop', 'average', 'hann']
        if self.aggregation_mode not in valid_aggregation:
            raise ValueError(f"aggregation_mode must be one of {valid_aggregation}")
        
        # Validate patch_overlap (negative is meaningless; pixel overlap must be < patch_size)
        if isinstance(self.patch_overlap, (int, float)):
            if self.patch_overlap < 0:
                raise ValueError("patch_overlap cannot be negative")
            if self.patch_overlap >= 1:  # Pixel value, not fraction
                for ps in self.patch_size:
                    if self.patch_overlap >= ps:
                        raise ValueError(
                            f"patch_overlap ({self.patch_overlap}) must be less than patch_size ({ps}). "
                            f"Overlap >= patch_size creates step_size <= 0 (infinite patches)."
                        )
        elif isinstance(self.patch_overlap, (list, tuple)):
            for i, (overlap, ps) in enumerate(zip(self.patch_overlap, self.patch_size)):
                if overlap < 0:
                    raise ValueError(f"patch_overlap[{i}] cannot be negative")
                if overlap >= ps:
                    raise ValueError(
                        f"patch_overlap[{i}] ({overlap}) must be less than patch_size[{i}] ({ps}). "
                        f"Overlap >= patch_size creates step_size <= 0 (infinite patches)."
                    )

        non_div = [s for s in self.patch_size if s % _UNET_DIVISOR != 0]
        if non_div:
            warnings.warn(
                f"patch_size {self.patch_size} has dimensions not divisible by {_UNET_DIVISOR}. "
                f"Most encoder-decoder architectures (e.g., U-Net) require patch sizes "
                f"divisible by {_UNET_DIVISOR} (2^4 for 4 downsampling levels)."
            )

        if not 0.0 <= self.binary_threshold <= 1.0:
            raise ValueError(f"binary_threshold must be in [0, 1], got {self.binary_threshold}")

        # Coerce normalization (live transforms or specs) to JSON-serializable specs so the
        # config can round-trip through store_patch_variables/load_patch_variables.
        if self.normalization is not None:
            self.normalization = transforms_to_specs(self.normalization)

    @classmethod
    def from_dataset(
        cls,
        dataset: 'MedDataset',
        target_spacing: list = None,
        min_patch_size: list = None,
        max_patch_size: list = None,
        normalization: list = None,
        divisor: int = _UNET_DIVISOR,
        **kwargs
    ) -> 'PatchConfig':
        """Create PatchConfig with automatic patch_size from dataset analysis.

        Args:
            dataset: MedDataset instance with analyzed images.
            target_spacing: Target voxel spacing [x, y, z]. If None, uses
                dataset.get_suggestion()['target_spacing'].
            min_patch_size: Minimum per dimension [32, 32, 32].
            max_patch_size: Maximum per dimension [256, 256, 256].
            normalization: Optional list of normalization transforms (or specs) stored on the
                config as the single source of truth (e.g. [ZNormalization(masking_method='foreground')]).
            divisor: Divisibility constraint (default 16 for UNet compatibility).
            **kwargs: Additional PatchConfig parameters (samples_per_volume,
                sampler_type, label_probabilities, etc.).

        Returns:
            PatchConfig with suggested patch_size, apply_reorder, target_spacing.

        Example:
            >>> dataset = MedDataset(dataframe=df, mask_col='mask_path', dtype=MedMask)
            >>> config = PatchConfig.from_dataset(dataset, samples_per_volume=16)
        """
        suggestion = dataset.get_suggestion()
        _target_spacing = target_spacing if target_spacing is not None else suggestion['target_spacing']

        patch_size = suggest_patch_size(
            dataset,
            target_spacing=_target_spacing,
            min_patch_size=min_patch_size,
            max_patch_size=max_patch_size,
            divisor=divisor
        )

        # apply_reorder comes straight from the dataset, not get_suggestion() (it is not data-derived)
        config_kwargs = {
            'patch_size': patch_size,
            'apply_reorder': dataset.apply_reorder,
            'target_spacing': _target_spacing,
            'normalization': normalization,
        }
        config_kwargs.update(kwargs)

        return cls(**config_kwargs)

In [ ]:
config = PatchConfig(patch_size=[96, 96, 96], samples_per_volume=16)
test_eq(config.patch_size, [96, 96, 96])
test_eq(config.samples_per_volume, 16)
test_eq(config.sampler_type, 'uniform')
test_eq(config.apply_reorder, True)
test_eq(config.target_spacing, None)
test_eq(config.preprocessed, False)
test_eq(config.padding_mode, 0)

config2 = PatchConfig(
    patch_size=[64, 64, 64],
    apply_reorder=True,
    target_spacing=[0.5, 0.5, 0.5],
    padding_mode=0
)
test_eq(config2.apply_reorder, True)
test_eq(config2.target_spacing, [0.5, 0.5, 0.5])

# preprocessed=True with preprocessing params: no warning
config3 = PatchConfig(
    patch_size=[96, 96, 96],
    apply_reorder=True,
    target_spacing=[0.5, 0.5, 0.5],
    preprocessed=True
)
test_eq(config3.preprocessed, True)
test_eq(config3.apply_reorder, True)
test_eq(config3.target_spacing, [0.5, 0.5, 0.5])

# preprocessed=True without preprocessing params does NOT warn
# (preprocessed=True still has effect: skips pre_patch_tfms during training)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    config4 = PatchConfig(
        patch_size=[96, 96, 96],
        apply_reorder=False,
        target_spacing=None,
        preprocessed=True
    )
    preprocessed_warns = [x for x in w if 'preprocessed' in str(x.message).lower()]
    test_eq(len(preprocessed_warns), 0)

## Subject conversion

In [ ]:
#| export
def med_to_subject(
    img: Path | str,
    mask: Path | str = None,
) -> tio.Subject:
    """Create a TorchIO Subject with LAZY loading (stores paths only, no tensors).

    Storing only paths lets TorchIO's Queue workers load volumes on-demand during
    training, keeping RAM usage low.

    Args:
        img: Path to image file.
        mask: Path to mask file (optional).

    Returns:
        TorchIO Subject with 'image' and optionally 'mask' keys (lazy loaded).

    Example:
        >>> subject = med_to_subject('image.nii.gz', 'mask.nii.gz')
        >>> data = subject['image'].data  # volume loaded only now
    """
    subject_dict = {
        'image': tio.ScalarImage(path=str(img))  # Lazy - stores path only
    }
    
    if mask is not None:
        subject_dict['mask'] = tio.LabelMap(path=str(mask))
    
    return tio.Subject(**subject_dict)

In [ ]:
#| export
def create_subjects_dataset(
    df: pd.DataFrame,
    img_col: str,
    mask_col: str = None,
    pre_tfms: list = None,
    ensure_affine_consistency: bool = True
) -> tio.SubjectsDataset:
    """Build a TorchIO SubjectsDataset with LAZY loading from a DataFrame.

    Stores only file paths (not tensors); volumes are loaded on-demand by Queue
    workers, keeping memory usage constant regardless of dataset size.

    Args:
        df: DataFrame with image (and optionally mask) paths.
        img_col: Column name containing image paths.
        mask_col: Column name containing mask paths (optional).
        pre_tfms: List of TorchIO transforms to apply before patch extraction.
                  Use tio.ToCanonical() for reordering and tio.Resample() for resampling.
        ensure_affine_consistency: If True and mask_col is provided, automatically
            prepends tio.CopyAffine(target='image') to ensure spatial metadata
            consistency between image and mask. This prevents "More than one value
            for direction found" errors. Defaults to True.

    Returns:
        TorchIO SubjectsDataset with lazy-loaded subjects.

    Example:
        >>> pre_tfms = [tio.ToCanonical(), tio.Resample([0.5, 0.5, 0.5]), tio.ZNormalization()]
        >>> dataset = create_subjects_dataset(df, img_col='image', mask_col='label', pre_tfms=pre_tfms)
    """
    subjects = []
    for idx, row in df.iterrows():
        img_path = row[img_col]
        mask_path = row[mask_col] if mask_col else None
        subject = med_to_subject(img=img_path, mask=mask_path)
        subjects.append(subject)

    all_transforms = []

    # CopyAffine must run FIRST (mask present): aligns spatial metadata before other transforms
    if mask_col is not None and ensure_affine_consistency:
        all_transforms.append(tio.CopyAffine(target='image'))

    if pre_tfms:
        all_transforms.extend(pre_tfms)

    transform = tio.Compose(all_transforms) if all_transforms else None

    return tio.SubjectsDataset(subjects, transform=transform)

## Sampler creation

In [ ]:
#| export
def create_patch_sampler(config: PatchConfig) -> tio.data.PatchSampler:
    """Create appropriate TorchIO sampler based on config.
    
    Args:
        config: PatchConfig with sampler settings.
    
    Returns:
        TorchIO PatchSampler instance.
    
    Example:
        >>> config = PatchConfig(patch_size=[96, 96, 96], sampler_type='label')
        >>> sampler = create_patch_sampler(config)
    """
    patch_size = config.patch_size
    
    if config.sampler_type == 'uniform':
        return tio.UniformSampler(patch_size)
    
    elif config.sampler_type == 'label':
        return tio.LabelSampler(
            patch_size,
            label_name='mask',
            label_probabilities=config.label_probabilities
        )
    
    elif config.sampler_type == 'weighted':
        raise NotImplementedError(
            "WeightedSampler requires a pre-computed probability map which is not currently supported. "
            "Use 'label' sampler with label_probabilities for weighted sampling based on segmentation labels, "
            "or 'uniform' for random patch extraction."
        )
    
    raise ValueError(f"Unknown sampler type: {config.sampler_type}")

In [ ]:
config = PatchConfig(patch_size=[64, 64, 64], sampler_type='uniform')
sampler = create_patch_sampler(config)
test_eq(type(sampler), tio.UniformSampler)

# WeightedSampler raises NotImplementedError
from fastcore.test import test_fail
config_weighted = PatchConfig(patch_size=[64, 64, 64], sampler_type='weighted')
test_fail(lambda: create_patch_sampler(config_weighted), contains='WeightedSampler')

In [ ]:
device = _get_default_device()
test_eq(type(device), torch.device)

# _warn_config_override: no warning when values match or one is None
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _warn_config_override('test_param', True, True)  # Same values - no warning
    _warn_config_override('test_param', True, None)  # Explicit is None - no warning
    _warn_config_override('test_param', None, True)  # Config is None - no warning
    test_eq(len(w), 0)

# _warn_config_override warns when values differ
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _warn_config_override('test_param', True, False)  # Different values - warning
    test_eq(len(w), 1)
    assert 'mismatch' in str(w[0].message)

## Patch DataLoaders

In [ ]:
#| export
class MedPatchDataLoader:
    """DataLoader wrapper for patch-based training with TorchIO Queue.

    This class wraps a TorchIO Queue to provide a fastai-compatible DataLoader
    interface for patch-based training.

    Args:
        subjects_dataset: TorchIO SubjectsDataset.
        config: PatchConfig with queue and sampler settings.
        batch_size: Number of patches per batch. Must be positive.
        patch_tfms: Transforms to apply to extracted patches (training only).
            Accepts both fastMONAI wrappers (e.g., RandomAffine, RandomGamma) and
            raw TorchIO transforms. fastMONAI wrappers are automatically normalized
            to raw TorchIO for internal use. Mutually exclusive with gpu_augmentation.
        gpu_augmentation: GpuPatchAugmentation instance for GPU-batched augmentation.
            Operates on [B,C,D,H,W] tensors already on GPU, avoiding per-sample CPU
            overhead. Mutually exclusive with patch_tfms. Training only.
        shuffle: Whether to shuffle subjects and patches.
        drop_last: Whether to drop last incomplete batch.
    """

    def __init__(
        self,
        subjects_dataset: tio.SubjectsDataset,
        config: PatchConfig,
        batch_size: int = 4,
        patch_tfms: list = None,
        gpu_augmentation=None,
        shuffle: bool = True,
        drop_last: bool = False
    ):
        if batch_size <= 0:
            raise ValueError(f"batch_size must be positive, got {batch_size}")

        self.subjects_dataset = subjects_dataset
        self.config = config
        self.bs = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last
        self._device = _get_default_device()
        self.gpu_augmentation = gpu_augmentation

        self.sampler = create_patch_sampler(config)

        # Normalize accepts both fastMONAI wrappers and raw TorchIO transforms
        normalized_tfms = normalize_patch_transforms(patch_tfms)
        self.patch_tfms = tio.Compose(normalized_tfms) if normalized_tfms else None

        self.queue = tio.Queue(
            subjects_dataset,
            max_length=config.queue_length,
            samples_per_volume=config.samples_per_volume,
            sampler=self.sampler,
            num_workers=config.queue_num_workers,
            shuffle_subjects=shuffle,
            shuffle_patches=shuffle
        )

        self._dl = DataLoader(
            self.queue,
            batch_size=batch_size,
            num_workers=0,  # Queue handles workers
            drop_last=drop_last
        )

        self._closed = False

    def __iter__(self):
        """Iterate over batches, yielding (image, mask) tuples."""
        for batch in self._dl:
            img = batch['image'][tio.DATA]  # [B, C, H, W, D]
            has_mask = 'mask' in batch

            # Apply CPU patch transforms if provided (per-sample TorchIO loop)
            if self.patch_tfms is not None:
                img, mask = self._apply_patch_tfms(batch, has_mask)
            else:
                mask = batch['mask'][tio.DATA] if has_mask else None

            img = img.to(self._device)
            if mask is not None:
                mask = mask.to(self._device)

            # Apply GPU augmentation if provided (batched, on-device)
            if self.gpu_augmentation is not None:
                img, mask = self.gpu_augmentation(img, mask)

            img = MedImage(img)
            if mask is not None:
                mask = MedMask(mask)

            yield img, mask

    def _apply_patch_tfms(self, batch, has_mask):
        """Apply per-sample CPU TorchIO patch transforms; returns stacked (img, mask|None)."""
        transformed_imgs = []
        transformed_masks = [] if has_mask else None
        for i in range(batch['image'][tio.DATA].shape[0]):
            subject_dict = {'image': tio.ScalarImage(tensor=batch['image'][tio.DATA][i])}
            if has_mask:
                subject_dict['mask'] = tio.LabelMap(tensor=batch['mask'][tio.DATA][i])
            subject = tio.Subject(subject_dict)
            transformed = self.patch_tfms(subject)
            transformed_imgs.append(transformed['image'].data)
            if has_mask:
                transformed_masks.append(transformed['mask'].data)
        img = torch.stack(transformed_imgs)
        mask = torch.stack(transformed_masks) if has_mask else None
        return img, mask

    def __len__(self):
        """Return number of batches per epoch."""
        n_patches = len(self.subjects_dataset) * self.config.samples_per_volume
        if self.drop_last:
            return n_patches // self.bs
        return (n_patches + self.bs - 1) // self.bs

    @property
    def dataset(self):
        """Return the underlying queue as dataset."""
        return self.queue

    @property
    def device(self):
        """Return current device."""
        return self._device

    def to(self, device):
        """Move DataLoader to device."""
        self._device = device
        return self

    def one_batch(self):
        """Return one batch from the DataLoader.

        Required for fastai compatibility - used for device detection
        and batch shape validation during Learner initialization.

        Returns:
            Tuple of (image, mask) tensors on the correct device.
        """
        return next(iter(self))

    def close(self):
        """Shut down TorchIO Queue workers. Safe to call multiple times."""
        if self._closed:
            return
        self._closed = True
        try:
            # Drop the Queue's internal iterator so its worker DataLoader shuts down
            if hasattr(self, 'queue') and hasattr(self.queue, '_subjects_iterable'):
                self.queue._subjects_iterable = None
        except Exception:
            pass

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()
        return False

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

In [ ]:
#| export
def _split_df(df, valid_pct, valid_col, seed):
    """Split a DataFrame into (train_df, valid_df) by valid_col, else a random valid_pct split."""
    if valid_col is not None:
        train_df = df[df[valid_col] == False].reset_index(drop=True)
        valid_df = df[df[valid_col] == True].reset_index(drop=True)
    else:
        rng = np.random.RandomState(seed)
        n = len(df)
        valid_idx = rng.choice(n, size=int(n * valid_pct), replace=False)
        train_idx = np.setdiff1d(np.arange(n), valid_idx)
        train_df = df.iloc[train_idx].reset_index(drop=True)
        valid_df = df.iloc[valid_idx].reset_index(drop=True)
    return train_df, valid_df


def _build_pre_patch_tfms(patch_config, apply_reorder, target_spacing, pre_patch_tfms):
    """Assemble the pre-patch TorchIO transform list (reorder/resample/user), honoring preprocessed.

    Returns None when no transforms apply (matches passing pre_tfms=None to create_subjects_dataset).
    """
    if patch_config.preprocessed:
        return None
    all_pre_tfms = []
    if apply_reorder:
        all_pre_tfms.append(tio.ToCanonical())
    if target_spacing is not None:
        all_pre_tfms.append(tio.Resample(target_spacing))
    if pre_patch_tfms:
        all_pre_tfms.extend(normalize_patch_transforms(pre_patch_tfms))
    return all_pre_tfms if all_pre_tfms else None


def _stash_from_df_metadata(instance, img_col, mask_col, pre_patch_tfms, apply_reorder,
                            target_spacing, ensure_affine_consistency, patch_config,
                            train_df, valid_df):
    """Record the de-facto metadata contract attributes consumed by research/vs_seg scripts.

    Stores the RAW pre_patch_tfms (not the normalized list), matching how the train_final*.py
    scripts set dls._pre_patch_tfms manually.
    """
    instance._img_col = img_col
    instance._mask_col = mask_col
    instance._pre_patch_tfms = pre_patch_tfms
    instance._apply_reorder = apply_reorder
    instance._target_spacing = target_spacing
    instance._ensure_affine_consistency = ensure_affine_consistency
    instance._patch_config = patch_config
    instance._train_source_df = train_df
    instance._valid_source_df = valid_df
    return instance


class MedPatchDataLoaders:
    """fastai-compatible DataLoaders for patch-based training with LAZY loading.

    This class provides train and validation DataLoaders that work with
    fastai's Learner for patch-based training on 3D medical images.

    Memory-efficient: Volumes are loaded on-demand by Queue workers,
    keeping memory usage constant (~150 MB) regardless of dataset size.

    Note: Validation uses the same sampling as training (pseudo Dice).
    For true validation metrics, use PatchInferenceEngine with GridSampler
    for full-volume sliding window inference.

    Example:
        >>> config = PatchConfig(patch_size=[96, 96, 96], apply_reorder=True, target_spacing=[0.5, 0.5, 0.5])
        >>> dls = MedPatchDataLoaders.from_df(
        ...     df, img_col='image', mask_col='label', valid_pct=0.2,
        ...     patch_config=config, pre_patch_tfms=[tio.ZNormalization()], bs=4
        ... )
        >>> learn = Learner(dls, model, loss_func=DiceLoss())
    """

    def __init__(
        self,
        train_dl: MedPatchDataLoader,
        valid_dl: MedPatchDataLoader,
        device: torch.device = None
    ):
        self._train_dl = train_dl
        self._valid_dl = valid_dl
        self._device = device or _get_default_device()

        self._train_dl.to(self._device)
        self._valid_dl.to(self._device)

        self._closed = False

    @classmethod
    def from_df(
        cls,
        df: pd.DataFrame,
        img_col: str,
        mask_col: str = None,
        valid_pct: float = 0.2,
        valid_col: str = None,
        patch_config: PatchConfig = None,
        pre_patch_tfms: list = None,
        patch_tfms: list = None,
        gpu_augmentation=None,
        apply_reorder: bool = None,
        target_spacing: list = None,
        bs: int = 4,
        seed: int = None,
        device: torch.device = None,
        ensure_affine_consistency: bool = True
    ) -> 'MedPatchDataLoaders':
        """Create train/valid DataLoaders from DataFrame with LAZY loading.

        Memory-efficient: Only file paths are stored at creation time.
        Volumes are loaded on-demand by Queue workers during training.

        Note: Both train and valid use the same sampling strategy from patch_config.
        This gives pseudo Dice during training. For true validation metrics,
        use PatchInferenceEngine with full-volume sliding window inference.

        Args:
            df: DataFrame with image paths.
            img_col: Column name for image paths.
            mask_col: Column name for mask paths.
            valid_pct: Fraction of data for validation.
            valid_col: Column name for train/valid split (if pre-defined).
            patch_config: PatchConfig instance. Preprocessing params (apply_reorder,
                target_spacing) can be set here for DRY usage with PatchInferenceEngine.
            pre_patch_tfms: Optional override for patch_config.normalization. TorchIO transforms
                applied before patch extraction (after reorder/resample), e.g.
                [ZNormalization(masking_method='foreground')]. Accepts fastMONAI wrappers and raw
                TorchIO transforms. When given, it is recorded into patch_config.normalization
                (best-effort) so it persists for inference. Prefer setting normalization on the
                PatchConfig. Skipped when preprocessed=True (include in preprocess_dataset()
                transforms instead).
            patch_tfms: TorchIO transforms applied to extracted patches (training only).
                Mutually exclusive with gpu_augmentation.
            gpu_augmentation: GpuPatchAugmentation instance for GPU-batched augmentation
                (training only). Mutually exclusive with patch_tfms.
            apply_reorder: If True, reorder to RAS+ orientation. If None, uses
                patch_config.apply_reorder. Explicit value overrides config.
            target_spacing: Target voxel spacing [x, y, z]. If None, uses
                patch_config.target_spacing. Explicit value overrides config.
            bs: Batch size.
            seed: Random seed for splitting.
            device: Device to use.
            ensure_affine_consistency: If True and mask_col is provided, automatically
                adds tio.CopyAffine(target='image') as the first transform to prevent
                spatial metadata mismatch errors. Defaults to True.

        Returns:
            MedPatchDataLoaders instance.

        Example:
            >>> # GPU augmentation (use patch_tfms=... instead for CPU TorchIO augmentation)
            >>> gpu_aug = gpu_patch_augmentations(config.patch_size, config.target_spacing)
            >>> dls = MedPatchDataLoaders.from_df(
            ...     df, img_col='image', mask_col='label',
            ...     patch_config=config, gpu_augmentation=gpu_aug, bs=4
            ... )
        """
        # Validate mutual exclusivity
        if gpu_augmentation is not None and patch_tfms is not None:
            raise ValueError(
                "Cannot use both gpu_augmentation and patch_tfms. "
                "gpu_augmentation operates on GPU tensors batch-wise, while "
                "patch_tfms uses per-sample CPU TorchIO transforms. Choose one."
            )

        if patch_config is None:
            patch_config = PatchConfig()

        # Explicit args override config (backward compatibility)
        _apply_reorder = apply_reorder if apply_reorder is not None else patch_config.apply_reorder
        _target_spacing = target_spacing if target_spacing is not None else patch_config.target_spacing

        _warn_config_override('apply_reorder', patch_config.apply_reorder, apply_reorder)
        _warn_config_override('target_spacing', patch_config.target_spacing, target_spacing)

        train_df, valid_df = _split_df(df, valid_pct, valid_col, seed)

        # Normalization: patch_config.normalization is the source of truth. An explicit
        # pre_patch_tfms overrides it and is recorded into the config (best-effort) so it
        # persists for inference; non-serializable transforms are applied but not recorded.
        if pre_patch_tfms is not None:
            _norm_tfms = pre_patch_tfms
            try:
                _specs = transforms_to_specs(pre_patch_tfms)
            except TypeError as e:
                warnings.warn(f"pre_patch_tfms not recorded in patch_config.normalization: {e}")
            else:
                if patch_config.normalization and patch_config.normalization != _specs:
                    warnings.warn("pre_patch_tfms overrides patch_config.normalization.")
                patch_config.normalization = _specs
        else:
            _norm_tfms = transforms_from_specs(patch_config.normalization)

        # Skipped when patch_config.preprocessed
        all_pre_tfms = _build_pre_patch_tfms(patch_config, _apply_reorder, _target_spacing, _norm_tfms)

        # Lazy subjects datasets (paths only)
        train_subjects = create_subjects_dataset(
            train_df, img_col, mask_col,
            pre_tfms=all_pre_tfms,
            ensure_affine_consistency=ensure_affine_consistency
        )
        valid_subjects = create_subjects_dataset(
            valid_df, img_col, mask_col,
            pre_tfms=all_pre_tfms,
            ensure_affine_consistency=ensure_affine_consistency
        )

        # Train and valid share patch_config for consistent sampling
        train_dl = MedPatchDataLoader(
            train_subjects, patch_config, bs,
            patch_tfms=patch_tfms,
            gpu_augmentation=gpu_augmentation,
            shuffle=True, drop_last=True
        )
        valid_dl = MedPatchDataLoader(
            valid_subjects, patch_config, bs,
            patch_tfms=None,
            gpu_augmentation=None,
            shuffle=False, drop_last=False
        )

        instance = cls(train_dl, valid_dl, device)
        return _stash_from_df_metadata(
            instance, img_col, mask_col, _norm_tfms, _apply_reorder, _target_spacing,
            ensure_affine_consistency, patch_config, train_df, valid_df
        )

    @property
    def train(self):
        """Training DataLoader."""
        return self._train_dl

    @property
    def valid(self):
        """Validation DataLoader."""
        return self._valid_dl

    @property
    def train_ds(self):
        """Training subjects dataset."""
        return self._train_dl.subjects_dataset

    @property
    def valid_ds(self):
        """Validation subjects dataset."""
        return self._valid_dl.subjects_dataset

    @property
    def device(self):
        """Current device."""
        return self._device

    @property
    def bs(self):
        """Batch size."""
        return self._train_dl.bs

    @property
    def apply_reorder(self):
        """Whether reordering to RAS+ is enabled."""
        return getattr(self, '_apply_reorder', False)

    @property
    def target_spacing(self):
        """Target voxel spacing for resampling."""
        return getattr(self, '_target_spacing', None)

    @property
    def patch_config(self):
        """The PatchConfig used for this DataLoaders."""
        return getattr(self, '_patch_config', None)

    @property
    def split_df(self):
        """DataFrame recording train/valid split for reproducibility logging."""
        train = self._train_source_df.assign(is_valid=False)
        valid = self._valid_source_df.assign(is_valid=True)
        return pd.concat([train, valid], ignore_index=True)

    def to(self, device):
        """Move DataLoaders to device."""
        self._device = device
        self._train_dl.to(device)
        self._valid_dl.to(device)
        return self

    def __iter__(self):
        """Iterate over training DataLoader."""
        return iter(self._train_dl)

    def one_batch(self):
        """Return one batch from the training DataLoader.

        Required for fastai Learner compatibility - used for device
        detection and batch shape validation.
        """
        return self._train_dl.one_batch()

    def __len__(self):
        """Return number of batches in training DataLoader."""
        return len(self._train_dl)

    def __getitem__(self, idx):
        """Get DataLoader by index. Required for fastai Learner compatibility.

        Args:
            idx: 0 for training DataLoader, 1 for validation DataLoader.

        Returns:
            MedPatchDataLoader instance.
        """
        if idx == 0:
            return self._train_dl
        elif idx == 1:
            return self._valid_dl
        else:
            raise IndexError(f"Index {idx} out of range. Use 0 (train) or 1 (valid).")

    def cuda(self):
        """Move DataLoaders to CUDA device."""
        return self.to(torch.device('cuda'))

    def cpu(self):
        """Move DataLoaders to CPU."""
        return self.to(torch.device('cpu'))

    def show_batch(self, dl_idx=0, max_n=6, figsize=None, channel=0,
                   slice_index=None, anatomical_plane=0, overlay=False,
                   voxel_size=None, **kwargs):
        """Show a batch of patch samples for visualization."""

        dl = self[dl_idx]
        x, y = dl.one_batch()
        x = x.cpu()
        if y is not None: y = y.cpu()

        nrows = min(x.shape[0], max_n)
        has_mask = y is not None

        if overlay and has_mask:
            ncols = x.shape[1]
        else:
            ncols = x.shape[1] + (1 if has_mask else 0)

        if figsize is None:
            figsize = (ncols * 3, nrows * 3)
        fig, axs = plt.subplots(nrows, ncols, figsize=figsize, squeeze=False)
        flat_axs = axs.flatten()

        imgs, masks_for_overlay, slice_idxs = [], [], []
        for i in range(nrows):
            img = x[i]
            im_channels = [MedImage(c_img[None]) for c_img in img]

            if has_mask:
                mask = y[i]
                idx = find_max_slice(mask[0].numpy(), anatomical_plane) if slice_index is None else slice_index
                if overlay:
                    masks_for_overlay.extend([MedMask(mask)] * len(im_channels))
                else:
                    im_channels.append(MedMask(mask))
            else:
                idx = slice_index

            imgs.extend(im_channels)
            slice_idxs.extend([idx] * len(im_channels))

        _voxel_size = voxel_size if voxel_size is not None else self.target_spacing
        ctxs = [im.show(ax=ax, slice_index=idx, anatomical_plane=anatomical_plane,
                        voxel_size=_voxel_size)
                for im, ax, idx in zip(imgs, flat_axs, slice_idxs)]

        if overlay and has_mask:
            for mask, ax, idx in zip(masks_for_overlay, flat_axs, slice_idxs):
                mask.show(ax=ax, slice_index=idx, anatomical_plane=anatomical_plane,
                          voxel_size=_voxel_size)

        plt.tight_layout()
        plt.show()

    def new_empty(self):
        """Create a new empty version of self for learner export.

        Required for fastai Learner.export() compatibility - creates a
        lightweight placeholder that can be pickled without the full dataset.

        Returns:
            A minimal MedPatchDataLoaders-like object with no data.
        """
        class EmptyMedPatchDataLoaders:
            """Minimal placeholder for exported learner."""
            def __init__(self, device):
                self._device = device
            @property
            def device(self): return self._device
            def to(self, device):
                self._device = device
                return self
            def cpu(self):
                """Move to CPU. Required for load_learner compatibility."""
                return self.to(torch.device('cpu'))
            def new_empty(self):
                """Return self since already empty."""
                return self

        return EmptyMedPatchDataLoaders(self._device)

    def close(self):
        """Shut down all DataLoader workers. Safe to call multiple times."""
        if self._closed:
            return
        self._closed = True
        if hasattr(self, '_train_dl') and self._train_dl is not None:
            self._train_dl.close()
        if hasattr(self, '_valid_dl') and self._valid_dl is not None:
            self._valid_dl.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()
        return False

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

In [ ]:
#| hide
# _split_df must not touch the global numpy RNG; split is reproducible per seed.
np.random.seed(0); _s0 = np.random.get_state()
_split_df(pd.DataFrame({'x': range(10)}), valid_pct=0.2, valid_col=None, seed=42)
_s1 = np.random.get_state()
test_eq(_s0[0] == _s1[0] and np.array_equal(_s0[1], _s1[1]) and _s0[2] == _s1[2], True)
_a = _split_df(pd.DataFrame({'x': range(10)}), 0.2, None, seed=42)[1]['x'].tolist()
_b = _split_df(pd.DataFrame({'x': range(10)}), 0.2, None, seed=42)[1]['x'].tolist()
test_eq(_a, _b)

In [ ]:
from fastMONAI.vision_augmentation import GpuPatchAugmentation

# Both gpu_augmentation and patch_tfms -> ValueError
test_fail(
    lambda: MedPatchDataLoaders.from_df(
        pd.DataFrame({'img': ['fake.nii'], 'mask': ['fake.nii']}),
        img_col='img', mask_col='mask',
        patch_tfms=[tio.RandomFlip()],
        gpu_augmentation=GpuPatchAugmentation(flip={'axes': (0,), 'p': 0.5}),
    ),
    contains='Cannot use both'
)

# Verify gpu_augmentation is stored on train_dl but not valid_dl
# (We can't fully instantiate from_df without real files, so test MedPatchDataLoader directly)
test_eq(MedPatchDataLoader.__init__.__code__.co_varnames[:8],
        ('self', 'subjects_dataset', 'config', 'batch_size',
         'patch_tfms', 'gpu_augmentation', 'shuffle', 'drop_last'))

## Patch-based Inference

In [ ]:
#| export
import numbers

def _normalize_patch_overlap(patch_overlap: 'int | float | list', patch_size: list) -> tuple:
    """Convert patch_overlap (fraction 0-1, pixel int, numpy scalar, or sequence) to a tuple of even pixel ints for TorchIO's GridSampler.

    Format conversion only; value validation (negatives, overlap >= patch_size) lives in PatchConfig.__post_init__().
    """
    # Scalar fraction (0 < x < 1); excludes 1.0 since 100% overlap creates step_size=0 (infinite patches)
    if isinstance(patch_overlap, (int, float, numbers.Number)) and 0 < float(patch_overlap) < 1:
        result = []
        for ps in patch_size:
            pixels = int(int(ps) * float(patch_overlap))
            if pixels % 2 != 0:  # TorchIO requires even overlap
                pixels = pixels - 1 if pixels > 0 else 0
            result.append(pixels)
        return tuple(result)

    # Scalar int (incl. numpy scalars): pixel count
    if isinstance(patch_overlap, (int, float, numbers.Number)):
        val = int(patch_overlap)
        if val % 2 != 0:
            val = val - 1 if val > 0 else 0
        return tuple(val for _ in patch_size)

    # Sequence (list, tuple, ndarray)
    result = []
    for val in patch_overlap:
        pixels = int(val)
        if pixels % 2 != 0:
            pixels = pixels - 1 if pixels > 0 else 0
        result.append(pixels)
    return tuple(result)


# Batch tensor shape: [B, C, D, H, W], spatial dims are 2, 3, 4.
_TTA_FLIP_AXES = (
    (),         # original
    (4,),       # flip LR (W)
    (3,),       # flip AP (H)
    (2,),       # flip IS (D)
    (3, 4),     # flip LR+AP
    (2, 4),     # flip LR+IS
    (2, 3),     # flip AP+IS
    (2, 3, 4),  # flip all
)


def _logits_to_probs(logits: torch.Tensor) -> torch.Tensor:
    """Logits -> probabilities: sigmoid for 1 channel (binary), else softmax over the channel dim.

    Upcasts to float32 (logits may be bfloat16 under AMP). Does NOT move to CPU; callers decide
    placement (TTA keeps tensors on-device to flip back; the non-TTA path moves to CPU immediately).
    """
    return torch.sigmoid(logits.float()) if logits.shape[1] == 1 else torch.softmax(logits.float(), dim=1)


def _predict_patch_tta(models, patch_input, amp_context=None) -> torch.Tensor:
    """Mirror TTA (with optional model ensemble): average probabilities over the 8 flips and all models.

    models is a single model or a list of models; a list soft-votes (averages probabilities).
    Uses a running sum (2x memory, not 9x per model). Each pass: flip -> forward -> activate ->
    flip back -> accumulate. Returns the averaged probability tensor [B, C, D, H, W] on CPU.
    amp_context, if given, wraps each forward pass.
    """
    if amp_context is None: amp_context = nullcontext()
    if not isinstance(models, (list, tuple)):
        models = [models]
    summed_probs = None
    for axes in _TTA_FLIP_AXES:
        flipped = torch.flip(patch_input, list(axes)) if axes else patch_input
        for model in models:
            with amp_context:
                logits = model(flipped)
            probs = _logits_to_probs(logits)
            if axes:
                probs = torch.flip(probs, list(axes))
            summed_probs = probs if summed_probs is None else summed_probs + probs
    return (summed_probs / (len(_TTA_FLIP_AXES) * len(models))).cpu()


@dataclass
class _PreparedSubject:
    """Intermediate state from image preparation, used for pipelined inference."""
    subject: tio.Subject
    org_img: tio.Image
    input_img: tio.Image
    org_size: tuple
    grid_sampler: tio.GridSampler
    aggregator: tio.GridAggregator
    patch_loader: DataLoader


class PatchInferenceEngine:
    """Patch-based inference with automatic volume reconstruction.
    
    Uses TorchIO's GridSampler to extract overlapping patches and
    GridAggregator to reconstruct the full volume from predictions.
    
    Args:
        learner: fastai Learner or PyTorch model (nn.Module), or a list of them for
            soft-vote ensembling (per-patch probabilities are averaged before argmax/
            threshold). When passing a raw PyTorch model, load weights first with
            model.load_state_dict(). A torch.compile'd model is unwrapped for inference;
            the caller's wrapper object is left in place.
        config: PatchConfig with inference settings. Preprocessing params (apply_reorder,
            target_spacing, padding_mode) can be set here for DRY usage.
        apply_reorder: Whether to reorder to RAS+ orientation. If None, uses config value.
        target_spacing: Target voxel spacing. If None, uses config value.
        sw_batch_size: Number of patches to predict at once. Must be positive.
        pre_inference_tfms: Optional override for config.normalization. If None, normalization is
            read from config.normalization (the default source of truth, set at training time).
            Provide this only to override the config (e.g. for non-serializable transforms).
            Accepts both fastMONAI wrappers and raw TorchIO transforms.
        amp: If True, use automatic mixed precision (bfloat16) for the forward pass.
            Only supported on CUDA devices; ignored with a warning on CPU/MPS.
            Defaults to False.
    
    Example:
        >>> # normalization is read from config.normalization; amp=True for faster GPU
        >>> config = PatchConfig(patch_size=[96, 96, 96], normalization=[ZNormalization(masking_method='foreground')])
        >>> engine = PatchInferenceEngine(learn, config)
        >>> pred = engine.predict('image.nii.gz')
    """
    
    def __init__(
        self,
        learner,
        config: PatchConfig,
        apply_reorder: bool = None,
        target_spacing: list = None,
        sw_batch_size: int = 4,
        pre_inference_tfms: list = None,
        amp: bool = False
    ):
        if sw_batch_size <= 0:
            raise ValueError(f"sw_batch_size must be positive, got {sw_batch_size}")
        
        # We check for Learner explicitly because some models (e.g., MONAI UNet) have a
        # .model attribute that is NOT the full model but an internal Sequential.
        _learners = list(learner) if isinstance(learner, (list, tuple)) else [learner]
        if len(_learners) == 0:
            raise ValueError("learner must be a model/Learner or a non-empty list of them.")
        _models = [l.model if isinstance(l, Learner) else l for l in _learners]
        self.models = [unwrap_compiled_model(m) for m in _models]
        if any(u is not m for u, m in zip(self.models, _models)):
            warnings.warn(
                "torch.compile wrapper removed; patch inference runs the underlying "
                "module eagerly, so compile speedups do not apply."
            )
        self.model = self.models[0]  # first model; alias kept for single-model back-compat
        
        self.config = config
        self.sw_batch_size = sw_batch_size
        
        # Normalization: explicit pre_inference_tfms overrides config.normalization (the
        # default source of truth). Accepts fastMONAI wrappers and raw TorchIO transforms.
        _tfms = (pre_inference_tfms if pre_inference_tfms is not None
                 else transforms_from_specs(getattr(config, 'normalization', None)))
        normalized_tfms = normalize_patch_transforms(_tfms)
        self.pre_inference_tfms = tio.Compose(normalized_tfms) if normalized_tfms else None
        
        # Explicit args override config (backward compatibility)
        self.apply_reorder = apply_reorder if apply_reorder is not None else config.apply_reorder
        self.target_spacing = target_spacing if target_spacing is not None else config.target_spacing
        
        _warn_config_override('apply_reorder', config.apply_reorder, apply_reorder)
        _warn_config_override('target_spacing', config.target_spacing, target_spacing)
        
        # Get device from model: check explicit .device attribute first (custom wrappers),
        # then fall back to parameter inspection (nn.Module)
        if hasattr(self.model, 'device'):
            self._device = torch.device(self.model.device)
        else:
            try:
                self._device = next(self.model.parameters()).device
            except StopIteration:
                self._device = _get_default_device()

        # Set eval mode once at construction (this class is inference-only)
        for _m in self.models:
            _m.eval()

        # AMP: CUDA-only, matching nnU-Net pattern
        if amp and self._device.type == 'cuda':
            self._amp_context = torch.amp.autocast('cuda', dtype=torch.bfloat16)
        else:
            if amp and self._device.type != 'cuda':
                warnings.warn("AMP is only supported on CUDA devices. Ignoring amp=True.")
            self._amp_context = nullcontext()

    def _prepare_subject(self, img_path: Path | str) -> _PreparedSubject:
        """Load + preprocess the image and build its GridSampler/Aggregator/DataLoader.

        Thread-safe: creates only new local objects and reads only immutable self config.
        """
        # Keep org_img and org_size for post-processing
        org_img, input_img, org_size = med_img_reader(
            img_path, apply_reorder=self.apply_reorder, target_spacing=self.target_spacing, only_tensor=False
        )

        subject = tio.Subject(
            image=tio.ScalarImage(tensor=input_img.data.float(), affine=input_img.affine)
        )

        # Match training preprocessing (e.g., ZNormalization)
        if self.pre_inference_tfms is not None:
            subject = self.pre_inference_tfms(subject)

        # Pad dimensions smaller than patch_size, keep larger dimensions intact
        img_shape = subject['image'].shape[1:]  # Exclude channel dim
        target_size = [max(s, p) for s, p in zip(img_shape, self.config.patch_size)]

        if any(s < p for s, p in zip(img_shape, self.config.patch_size)):
            padded_dims = [f"dim{i}: {s}<{p}" for i, (s, p) in enumerate(zip(img_shape, self.config.patch_size)) if s < p]
            warnings.warn(
                f"Image size {list(img_shape)} smaller than patch_size {self.config.patch_size} "
                f"in {padded_dims}. Padding with mode={self.config.padding_mode}. "
                "Ensure training data covered similar sizes to avoid artifacts."
            )

        subject = tio.CropOrPad(target_size, padding_mode=self.config.padding_mode)(subject)

        patch_overlap = _normalize_patch_overlap(self.config.patch_overlap, self.config.patch_size)

        grid_sampler = tio.GridSampler(
            subject, patch_size=self.config.patch_size, patch_overlap=patch_overlap
        )
        aggregator = tio.GridAggregator(
            grid_sampler, overlap_mode=self.config.aggregation_mode
        )
        patch_loader = DataLoader(grid_sampler, batch_size=self.sw_batch_size, num_workers=0)

        return _PreparedSubject(
            subject=subject, org_img=org_img, input_img=input_img,
            org_size=org_size, grid_sampler=grid_sampler,
            aggregator=aggregator, patch_loader=patch_loader
        )

    def _run_inference(self, prepared: _PreparedSubject, tta: bool = False) -> torch.Tensor:
        """Run the model(s) over all patches and aggregate; returns the raw probability tensor.

        Must run on the main thread (model forward pass). tta enables mirror test-time augmentation.
        With multiple models the per-patch probabilities are averaged (soft voting).
        """
        # inference_mode is slightly faster than no_grad (disables autograd tracking
        # and view tracking). Safe here since we don't do in-place ops on outputs.
        with torch.inference_mode():
            for patches_batch in prepared.patch_loader:
                patch_input = patches_batch['image'][tio.DATA].to(self._device)
                locations = patches_batch[tio.LOCATION]

                if tta:
                    probs = _predict_patch_tta(self.models, patch_input, self._amp_context)
                else:
                    summed = None
                    for _m in self.models:
                        with self._amp_context:
                            logits = _m(patch_input)
                        p = _logits_to_probs(logits)
                        summed = p if summed is None else summed + p
                    probs = (summed / len(self.models)).cpu()

                prepared.aggregator.add_batch(probs, locations)

        return prepared.aggregator.get_output_tensor()

    def _postprocess(
        self,
        output: torch.Tensor,
        prepared: _PreparedSubject,
        return_probabilities: bool = False
    ) -> tuple[torch.Tensor, np.ndarray]:
        """Post-process the aggregated output (threshold/argmax, resize, reorient); returns (result, affine).

        return_probabilities keeps the probability map instead of taking argmax/threshold.
        """
        if return_probabilities:
            result = output
        else:
            n_classes = output.shape[0]
            if n_classes == 1:
                result = (output >= self.config.binary_threshold).float()
            else:
                result = output.argmax(dim=0, keepdim=True).float()

        if not return_probabilities and self.config.keep_largest_component:
            if n_classes > 2:
                warnings.warn(
                    f"keep_largest_component is binary-only; skipping it for multi-class output "
                    f"(n_classes={n_classes}). Set keep_largest_component=False for multi-class "
                    f"models, or post-process each class separately."
                )
            else:
                from fastMONAI.vision_inference import keep_largest
                result = keep_largest(result.squeeze(0)).unsqueeze(0)

        if return_probabilities:
            pred_img = tio.ScalarImage(tensor=result.float(), affine=prepared.input_img.affine)
        else:
            pred_img = tio.LabelMap(tensor=result.float(), affine=prepared.input_img.affine)

        # Resize back to original size (before resampling)
        pred_img = _do_resize(pred_img, prepared.org_size, image_interpolation='nearest')

        # Reorient to original orientation (if reorder was applied)
        if self.apply_reorder:
            target_orientation = ''.join(prepared.org_img.orientation)
            pred_img = tio.ToOrientation(target_orientation)(pred_img)

        result = pred_img.data.cpu()
        if not return_probabilities:
            result = result.long()

        # Use original affine matrix for correct spatial alignment
        if not (hasattr(prepared.org_img, 'affine') and prepared.org_img.affine is not None):
            raise RuntimeError(
                "org_img.affine not available. This should never happen - please report this bug."
            )
        affine = prepared.org_img.affine.copy()

        return result, affine

    def predict(
        self,
        img_path: Path | str,
        return_probabilities: bool = False,
        return_affine: bool = False,
        tta: bool = False
    ) -> torch.Tensor | tuple[torch.Tensor, np.ndarray]:
        """Predict on a single volume using patch-based inference.

        Args:
            img_path: Path to input image.
            return_probabilities: If True, return probability map instead of argmax.
            return_affine: If True, return (prediction, affine) tuple instead of just prediction.
            tta: If True, apply mirror test-time augmentation (8 flip combinations, averaged
                probabilities; ~8x slower). Works best when training used RandomFlip. Defaults to False.

        Returns:
            Predicted segmentation mask tensor, or tuple (prediction, affine) if return_affine=True.
        """
        prepared = self._prepare_subject(img_path)
        output = self._run_inference(prepared, tta=tta)
        result, affine = self._postprocess(output, prepared, return_probabilities)

        if return_affine:
            return result, affine
        return result
    
    def to(self, device):
        """Move engine (all models) to device."""
        self._device = device
        for _m in self.models:
            _m.to(device)
        return self

In [ ]:
# PatchInferenceEngine must store a raw model as-is, not its internal .model
from monai.networks.nets import UNet
from monai.networks.layers import Norm

test_model = UNet(
    spatial_dims=3, in_channels=1, out_channels=2,
    channels=(16, 32), strides=(2,), num_res_units=1,
    norm=Norm.INSTANCE
)
test_config = PatchConfig(patch_size=[32, 32, 32])

# MONAI UNet has a .model attribute (the bug scenario)
test_eq(hasattr(test_model, 'model'), True)
test_eq(type(test_model.model).__name__, 'Sequential')  # It's an internal Sequential

# Raw model -> engine.model is the UNet, NOT unet.model
engine = PatchInferenceEngine(test_model, test_config)
test_eq(type(engine.model), UNet)  # Should be UNet, not Sequential

print("PatchInferenceEngine raw model detection test passed!")

In [ ]:
#| export
from concurrent.futures import ThreadPoolExecutor


def _save_prediction(pred, affine, input_path, save_path, return_probabilities):
    """Save a single prediction as a NIfTI file (`<stem>_pred.nii[.gz]` in save_path).

    Module-level (no closure captures) so it is safe to run in a background save thread.
    return_probabilities saves a ScalarImage, otherwise a LabelMap.
    """
    input_path = Path(input_path)
    stem = input_path.stem
    if input_path.suffix == '.gz' and stem.endswith('.nii'):
        stem = stem[:-4]
        out_name = f"{stem}_pred.nii.gz"
    elif input_path.suffix == '.nii':
        out_name = f"{stem}_pred.nii"
    else:
        out_name = f"{stem}_pred.nii.gz"
    out_path = save_path / out_name

    if return_probabilities:
        pred_img = tio.ScalarImage(tensor=pred, affine=affine)
    else:
        pred_img = tio.LabelMap(tensor=pred, affine=affine)
    pred_img.save(out_path)


def _predict_one(engine, prepared, return_probabilities, tta):
    """Inference + postprocess on a prepared subject (no I/O); returns (result, affine).

    Shared by both patch_inference paths to keep them in lockstep; save scheduling stays at each call site.
    """
    output = engine._run_inference(prepared, tta=tta)
    return engine._postprocess(output, prepared, return_probabilities)


def patch_inference(
    learner,
    config: PatchConfig,
    file_paths: list,
    apply_reorder: bool = None,
    target_spacing: list = None,
    sw_batch_size: int = 4,
    return_probabilities: bool = False,
    progress: bool = True,
    save_dir: str = None,
    pre_inference_tfms: list = None,
    tta: bool = False,
    prefetch: bool = True,
    amp: bool = False
) -> list:
    """Batch patch-based inference on multiple volumes.

    When prefetch=True (default), overlaps I/O with compute: while the current
    image is being inferred, the next image is loaded and preprocessed in a
    background thread, and the previous result is saved in the background.
    This eliminates most I/O idle time, especially on GPU where CPU prep and
    GPU compute use different hardware.

    Args:
        learner: PyTorch model or fastai Learner, or a list of them to soft-vote
            ensemble (per-patch probabilities are averaged before argmax/threshold).
        config: PatchConfig with inference settings. Preprocessing params (apply_reorder,
            target_spacing) can be set here for DRY usage.
        file_paths: List of image paths.
        apply_reorder: Whether to reorder to RAS+ orientation. If None, uses config value.
        target_spacing: Target voxel spacing. If None, uses config value.
        sw_batch_size: Patches per batch.
        return_probabilities: Return probability maps.
        progress: Show progress bar.
        save_dir: Directory to save predictions as NIfTI files. If None, predictions are not saved.
        pre_inference_tfms: Optional override for config.normalization. If None, normalization is
            read from config.normalization (the source of truth set at training time). Provide this
            only to override the config (e.g. for non-serializable transforms).
        tta: If True, mirror TTA (8 flip combinations).
        prefetch: If True (default), overlap I/O with compute using a background
            thread for preparation and saving. Holds two subjects in memory
            simultaneously (current + next). Set to False for memory-constrained
            environments processing very large volumes.
        amp: If True, use automatic mixed precision (bfloat16) for the forward pass.
            Only supported on CUDA devices; ignored with a warning on CPU/MPS.

    Returns:
        List of predicted tensors.

    Example:
        >>> config = PatchConfig(patch_size=[96, 96, 96], apply_reorder=True, target_spacing=[0.4102, 0.4102, 1.5],
        ...                      normalization=[ZNormalization(masking_method='foreground')])
        >>> predictions = patch_inference(  # normalization auto-applied from config
        ...     learner=learn, config=config, file_paths=val_paths,
        ...     save_dir='predictions/patch_based', amp=True
        ... )
    """
    _apply_reorder = apply_reorder if apply_reorder is not None else config.apply_reorder
    _target_spacing = target_spacing if target_spacing is not None else config.target_spacing

    engine = PatchInferenceEngine(
        learner, config, _apply_reorder, _target_spacing, sw_batch_size, pre_inference_tfms,
        amp=amp
    )

    save_path = None
    if save_dir is not None:
        save_path = Path(save_dir)
        save_path.mkdir(parents=True, exist_ok=True)

    predictions = []
    desc = 'Patch inference (TTA)' if tta else 'Patch inference'
    n_files = len(file_paths)

    # Pipelined path: overlap I/O with compute
    if prefetch and n_files > 1:
        pbar = tqdm(total=n_files, desc=desc) if progress else None
        with ThreadPoolExecutor(max_workers=1) as pool:
            # Kick off preparation of the first image
            prefetch_future = pool.submit(engine._prepare_subject, file_paths[0])
            save_future = None

            for i in range(n_files):
                prepared = prefetch_future.result()

                # Start prefetching the next image (if any)
                if i + 1 < n_files:
                    prefetch_future = pool.submit(engine._prepare_subject, file_paths[i + 1])

                # Run inference on the main thread
                result, affine = _predict_one(engine, prepared, return_probabilities, tta)
                predictions.append(result)

                # Wait for previous save to complete before submitting a new one
                if save_future is not None:
                    save_future.result()

                if save_path is not None:
                    save_future = pool.submit(
                        _save_prediction, result, affine, file_paths[i],
                        save_path, return_probabilities
                    )

                if pbar is not None:
                    pbar.update(1)

            # Wait for final save
            if save_future is not None:
                save_future.result()

        if pbar is not None:
            pbar.close()

    # Sequential fallback: single image or prefetch disabled
    else:
        iterator = tqdm(file_paths, desc=desc) if progress else file_paths
        for path in iterator:
            prepared = engine._prepare_subject(path)
            pred, affine = _predict_one(engine, prepared, return_probabilities, tta)
            predictions.append(pred)
            if save_path is not None:
                _save_prediction(pred, affine, path, save_path, return_probabilities)

    return predictions

In [ ]:
# Test _TTA_FLIP_AXES and _predict_patch_tta
from itertools import combinations

# _TTA_FLIP_AXES has exactly 8 entries (2^3 combinations for 3 axes)
test_eq(len(_TTA_FLIP_AXES), 8)

# All 2^3 combinations present (each axis in {2,3,4} independently on/off)
expected_combos = set()
axes = [2, 3, 4]
for r in range(len(axes) + 1):
    for combo in combinations(axes, r):
        expected_combos.add(combo)
actual_combos = set(tuple(sorted(a)) for a in _TTA_FLIP_AXES)
test_eq(actual_combos, expected_combos)

# _predict_patch_tta output shape and probability range
import torch.nn as nn

class _SimpleConv(nn.Module):
    """Minimal model for TTA testing."""
    def __init__(self, out_channels):
        super().__init__()
        self.conv = nn.Conv3d(1, out_channels, 1)
    def forward(self, x):
        return self.conv(x)

# Binary case (1 output channel -> sigmoid)
model_bin = _SimpleConv(1).eval()
dummy_input = torch.randn(2, 1, 8, 8, 8)  # [B=2, C=1, D, H, W]
with torch.no_grad():
    tta_out = _predict_patch_tta(model_bin, dummy_input)
test_eq(tta_out.shape, torch.Size([2, 1, 8, 8, 8]))
assert tta_out.min() >= 0.0 and tta_out.max() <= 1.0, f"Probabilities out of range: [{tta_out.min()}, {tta_out.max()}]"

# Multi-class case (3 output channels -> softmax)
model_mc = _SimpleConv(3).eval()
with torch.no_grad():
    tta_out_mc = _predict_patch_tta(model_mc, dummy_input)
test_eq(tta_out_mc.shape, torch.Size([2, 3, 8, 8, 8]))
assert tta_out_mc.min() >= 0.0 and tta_out_mc.max() <= 1.0

# TTA on constant input matches single forward pass
# A constant tensor is invariant to flipping, so TTA should equal single pass
const_input = torch.ones(1, 1, 8, 8, 8) * 0.5
with torch.no_grad():
    single_logits = model_bin(const_input)
    single_probs = torch.sigmoid(single_logits).cpu()
    tta_probs = _predict_patch_tta(model_bin, const_input)
assert torch.allclose(single_probs, tta_probs, atol=1e-6), "TTA on constant input should match single forward pass"

print("TTA tests passed!")

In [ ]:
# Test _PreparedSubject and decomposed predict path
import tempfile, os, nibabel as nib
from monai.networks.nets import UNet
from monai.networks.layers import Norm

# Create a small synthetic NIfTI file for testing
_test_data = np.random.randn(32, 32, 32).astype(np.float32)
_test_affine = np.eye(4)
_test_nii = nib.Nifti1Image(_test_data, _test_affine)

with tempfile.TemporaryDirectory() as tmpdir:
    img_path = os.path.join(tmpdir, 'test_img.nii.gz')
    nib.save(_test_nii, img_path)

    _model = UNet(
        spatial_dims=3, in_channels=1, out_channels=2,
        channels=(16, 32), strides=(2,), num_res_units=1,
        norm=Norm.INSTANCE
    ).eval()
    _config = PatchConfig(patch_size=[32, 32, 32])
    _engine = PatchInferenceEngine(_model, _config, apply_reorder=False)

    # Test 1: _prepare_subject returns _PreparedSubject with expected attributes
    prepared = _engine._prepare_subject(img_path)
    assert isinstance(prepared, _PreparedSubject), "Should return _PreparedSubject"
    assert isinstance(prepared.subject, tio.Subject)
    assert isinstance(prepared.grid_sampler, tio.GridSampler)
    assert isinstance(prepared.aggregator, tio.GridAggregator)
    assert isinstance(prepared.patch_loader, DataLoader)
    assert prepared.org_size is not None

    # Test 2: Decomposed path equals predict() output
    pred_decomposed, affine_decomposed = _engine._postprocess(
        _engine._run_inference(
            _engine._prepare_subject(img_path)
        ),
        _engine._prepare_subject(img_path)
    )
    pred_predict, affine_predict = _engine.predict(img_path, return_affine=True)

    assert torch.equal(pred_decomposed, pred_predict), "Decomposed path should match predict()"
    assert np.array_equal(affine_decomposed, affine_predict), "Affine should match"

    # Test 3: prefetch=True produces identical results to prefetch=False
    paths = [img_path, img_path]  # Two copies to trigger pipeline path
    preds_prefetch = patch_inference(
        _model, _config, paths, apply_reorder=False, progress=False, prefetch=True
    )
    preds_sequential = patch_inference(
        _model, _config, paths, apply_reorder=False, progress=False, prefetch=False
    )
    assert len(preds_prefetch) == len(preds_sequential) == 2
    for p1, p2 in zip(preds_prefetch, preds_sequential):
        assert torch.equal(p1, p2), "prefetch=True should produce identical results"

    # Test 4: Error propagation -- file-not-found raises (not silently swallowed)
    test_fail(
        lambda: patch_inference(
            _model, _config, ['/nonexistent/file.nii.gz'],
            apply_reorder=False, progress=False, prefetch=True
        )
    )
    test_fail(
        lambda: patch_inference(
            _model, _config, [img_path, '/nonexistent/file.nii.gz'],
            apply_reorder=False, progress=False, prefetch=True
        )
    )

    # Test 5: Save pipeline works correctly with prefetch=True
    save_dir = os.path.join(tmpdir, 'preds')
    preds_saved = patch_inference(
        _model, _config, paths, apply_reorder=False, progress=False,
        save_dir=save_dir, prefetch=True
    )
    assert len(preds_saved) == 2
    saved_files = list(Path(save_dir).glob('*.nii.gz'))
    assert len(saved_files) == 1, f"Expected 1 unique file (same input), got {len(saved_files)}"
    # Verify the saved file is valid NIfTI
    saved_nii = nib.load(str(saved_files[0]))
    assert saved_nii.shape is not None

print("Pipeline inference tests passed!")

In [ ]:
# Safety-net tests (added before refactor to characterize current behavior)
import tempfile as _tempfile, os as _os, nibabel as _nib
from monai.networks.nets import UNet as _UNet
from monai.networks.layers import Norm as _Norm

# --- T1: _normalize_patch_overlap covers zero / fraction / odd / int / list / numpy ---
test_eq(_normalize_patch_overlap(0, [96, 96, 96]), (0, 0, 0))
test_eq(_normalize_patch_overlap(0.5, [96, 96, 96]), (48, 48, 48))
test_eq(_normalize_patch_overlap(0.5, [98, 98, 98]), (48, 48, 48))   # 49 coerced down to even 48
test_eq(_normalize_patch_overlap(47, [64, 64, 64]), (46, 46, 46))    # odd int coerced even
test_eq(_normalize_patch_overlap([10, 11, 12], [64, 64, 64]), (10, 10, 12))
test_eq(_normalize_patch_overlap(np.int64(48), [96, 96, 96]), (48, 48, 48))

# --- T2: PatchConfig.__post_init__ validation + divisibility warning ---
test_fail(lambda: PatchConfig(sampler_type='bogus'), contains='sampler_type must be one of')
test_fail(lambda: PatchConfig(aggregation_mode='bogus'), contains='aggregation_mode must be one of')
test_fail(lambda: PatchConfig(patch_overlap=-1), contains='cannot be negative')
test_fail(lambda: PatchConfig(patch_overlap=[-1, 0, 0]), contains='cannot be negative')
test_fail(lambda: PatchConfig(patch_size=[64, 64, 64], patch_overlap=64), contains='must be less than patch_size')
test_fail(lambda: PatchConfig(patch_size=[64, 64, 64], patch_overlap=[10, 10, 64]), contains='must be less than patch_size')
_ok = PatchConfig(patch_size=[96, 96, 96], patch_overlap=0.5, sampler_type='label', aggregation_mode='hann')
test_eq(_ok.sampler_type, 'label')
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter('always')
    PatchConfig(patch_size=[90, 90, 90])
    test_eq(any('divisible by 16' in str(_x.message) for _x in _w), True)

# --- T3: return_probabilities=True returns a float probability map (production path, was untested) ---
with _tempfile.TemporaryDirectory() as _tmp:
    _ip = _os.path.join(_tmp, 'img.nii.gz')
    _nib.save(_nib.Nifti1Image(np.random.randn(32, 32, 32).astype(np.float32), np.eye(4)), _ip)
    _net = _UNet(spatial_dims=3, in_channels=1, out_channels=2, channels=(16, 32),
                 strides=(2,), num_res_units=1, norm=_Norm.INSTANCE).eval()
    _eng = PatchInferenceEngine(_net, PatchConfig(patch_size=[32, 32, 32]), apply_reorder=False)
    _prob = _eng.predict(_ip, return_probabilities=True)
    assert _prob.shape[0] == 2, f'expected 2 prob channels, got {tuple(_prob.shape)}'
    assert _prob.dtype.is_floating_point, _prob.dtype
    assert float(_prob.min()) >= 0.0 and float(_prob.max()) <= 1.0
    _mask = _eng.predict(_ip, return_probabilities=False)
    assert _mask.shape[0] == 1, f'expected 1 mask channel, got {tuple(_mask.shape)}'

# --- T4: from_df happy path (split, metadata contract, preprocessed skip) ---
def _tfm_names(ds):
    t = getattr(ds, 'transform', None)
    if t is None: t = getattr(ds, '_transform', None)
    if t is None: return []
    return [type(x).__name__ for x in getattr(t, 'transforms', [t])]

with _tempfile.TemporaryDirectory() as _tmp:
    _rows = []
    for _i in range(4):
        _ipath = _os.path.join(_tmp, f'img_{_i}.nii.gz')
        _mpath = _os.path.join(_tmp, f'msk_{_i}.nii.gz')
        _nib.save(_nib.Nifti1Image(np.random.randn(24, 24, 24).astype(np.float32), np.eye(4)), _ipath)
        _nib.save(_nib.Nifti1Image((np.random.rand(24, 24, 24) > 0.5).astype(np.uint8), np.eye(4)), _mpath)
        _rows.append({'img': _ipath, 'mask': _mpath, 'is_val': _i >= 2})
    _df = pd.DataFrame(_rows)
    _cfg = PatchConfig(patch_size=[16, 16, 16], samples_per_volume=2,
                       sampler_type='label', label_probabilities={0: 0.5, 1: 0.5})

    _dls = MedPatchDataLoaders.from_df(_df, img_col='img', mask_col='mask',
                                       valid_pct=0.5, patch_config=_cfg, seed=0, bs=1)
    test_eq(len(_dls.split_df), 4)
    test_eq(len(_dls._train_source_df), 2)
    test_eq(len(_dls._valid_source_df), 2)
    assert isinstance(_dls.train, MedPatchDataLoader) and isinstance(_dls.valid, MedPatchDataLoader)
    for _a in ['_img_col', '_mask_col', '_pre_patch_tfms', '_apply_reorder', '_target_spacing',
               '_ensure_affine_consistency', '_patch_config', '_train_source_df', '_valid_source_df']:
        assert hasattr(_dls, _a), f'from_df must set {_a}'
    test_eq(_dls.patch_config is _cfg, True)
    test_eq(_dls.apply_reorder, True)

    # valid_col split branch
    _dls2 = MedPatchDataLoaders.from_df(_df, img_col='img', mask_col='mask',
                                        valid_col='is_val', patch_config=_cfg, bs=1)
    test_eq(len(_dls2._train_source_df), 2)
    test_eq(len(_dls2._valid_source_df), 2)

    # preprocessed=True skips reorder/resample
    _cfg_pp = PatchConfig(patch_size=[16, 16, 16], samples_per_volume=2,
                          preprocessed=True, target_spacing=[1, 1, 1])
    _dls3 = MedPatchDataLoaders.from_df(_df, img_col='img', mask_col='mask',
                                        valid_pct=0.5, patch_config=_cfg_pp, seed=0, bs=1)
    _names_pp = _tfm_names(_dls3.train_ds)
    assert 'Resample' not in _names_pp and 'ToCanonical' not in _names_pp, f'preprocessed should skip, got {_names_pp}'

    # contrast: non-preprocessed WITH target_spacing includes Resample
    _cfg_rs = PatchConfig(patch_size=[16, 16, 16], samples_per_volume=2, target_spacing=[1, 1, 1])
    _dls4 = MedPatchDataLoaders.from_df(_df, img_col='img', mask_col='mask',
                                        valid_pct=0.5, patch_config=_cfg_rs, seed=0, bs=1)
    assert 'Resample' in _tfm_names(_dls4.train_ds)

# --- TH: MedPatchDataLoader.__iter__ with patch_tfms yields (MedImage, MedMask) (guards _apply_patch_tfms) ---
with _tempfile.TemporaryDirectory() as _tmp:
    _ip = _os.path.join(_tmp, 'i.nii.gz'); _mp = _os.path.join(_tmp, 'm.nii.gz')
    _nib.save(_nib.Nifti1Image(np.random.randn(20, 20, 20).astype(np.float32), np.eye(4)), _ip)
    _nib.save(_nib.Nifti1Image((np.random.rand(20, 20, 20) > 0.5).astype(np.uint8), np.eye(4)), _mp)
    _ds_h = create_subjects_dataset(pd.DataFrame({'img': [_ip], 'mask': [_mp]}), 'img', 'mask')
    _cfg_h = PatchConfig(patch_size=[16, 16, 16], samples_per_volume=2, queue_length=4, queue_num_workers=0)
    _dl_h = MedPatchDataLoader(_ds_h, _cfg_h, batch_size=2, patch_tfms=[tio.RandomFlip(flip_probability=0.0)])
    _xb, _yb = next(iter(_dl_h))
    assert isinstance(_xb, MedImage) and isinstance(_yb, MedMask), (type(_xb), type(_yb))
    test_eq(_xb.shape[0], 2)
    test_eq(tuple(_xb.shape[-3:]), (16, 16, 16))
    _dl_h.close()

# --- TB: binary_threshold uses >= (MONAI AsDiscrete semantics) and is configurable ---
with _tempfile.TemporaryDirectory() as _tmp:
    _ip = _os.path.join(_tmp, 'img.nii.gz')
    _nib.save(_nib.Nifti1Image(np.zeros((16, 16, 16), dtype=np.float32), np.eye(4)), _ip)
    _net1 = _UNet(spatial_dims=3, in_channels=1, out_channels=1, channels=(8, 16),
                  strides=(2,), num_res_units=1, norm=_Norm.INSTANCE).eval()
    # default threshold 0.5 with '>=': probability exactly 0.5 is foreground (would be background under '>')
    _eng_d = PatchInferenceEngine(_net1, PatchConfig(patch_size=[16, 16, 16]), apply_reorder=False)
    _prep = _eng_d._prepare_subject(_ip)
    _sp = _prep.input_img.spatial_shape
    _res_half = _eng_d._postprocess(torch.full((1, *_sp), 0.5), _prep, return_probabilities=False)[0]
    test_eq(int(_res_half.sum()), _res_half.numel())   # all 0.5 >= 0.5 -> all foreground
    # higher threshold makes 0.5 background, 0.7 foreground
    _eng_h = PatchInferenceEngine(_net1, PatchConfig(patch_size=[16, 16, 16], binary_threshold=0.7), apply_reorder=False)
    _prep_h = _eng_h._prepare_subject(_ip)
    _res_low = _eng_h._postprocess(torch.full((1, *_sp), 0.5), _prep_h, return_probabilities=False)[0]
    test_eq(int(_res_low.sum()), 0)                     # 0.5 < 0.7 -> background
    _res_at = _eng_h._postprocess(torch.full((1, *_sp), 0.7), _prep_h, return_probabilities=False)[0]
    test_eq(int(_res_at.sum()), _res_at.numel())        # 0.7 >= 0.7 -> foreground
test_fail(lambda: PatchConfig(binary_threshold=1.5), contains='binary_threshold')

print('Safety-net tests (T1-T4, TH, TB) passed!')

In [ ]:
#| hide
# --- PatchConfig.normalization: single source of truth for pre-patch/pre-inference norm ---
from fastMONAI.vision_augmentation import ZNormalization, _foreground_masking

# live transforms are coerced to JSON specs in __post_init__
_cfg = PatchConfig(patch_size=[96, 96, 96], normalization=[ZNormalization(masking_method='foreground')])
test_eq(_cfg.normalization, [{'name': 'ZNormalization', 'masking_method': 'foreground', 'channel_wise': True}])

# spec dicts pass through unchanged
_cfg2 = PatchConfig(patch_size=[96, 96, 96],
                    normalization=[{'name': 'ZNormalization', 'masking_method': 'foreground', 'channel_wise': True}])
test_eq(_cfg2.normalization, [{'name': 'ZNormalization', 'masking_method': 'foreground', 'channel_wise': True}])

# specs reconstruct to a live foreground z-norm
_rt = transforms_from_specs(_cfg.normalization)
test_eq(_rt[0].tio_transform.masking_method, _foreground_masking)

# PatchInferenceEngine reads config.normalization when pre_inference_tfms is None
_m = torch.nn.Conv3d(1, 2, 1)
_eng = PatchInferenceEngine(_m, _cfg)
test_eq(_eng.pre_inference_tfms is None, False)   # auto-applied from config
_eng_none = PatchInferenceEngine(_m, PatchConfig(patch_size=[96, 96, 96]))
test_eq(_eng_none.pre_inference_tfms, None)        # nothing configured -> None

# a non-serializable transform in config.normalization fails loudly
test_fail(lambda: PatchConfig(patch_size=[96, 96, 96], normalization=[tio.ZNormalization()]))

In [ ]:
# Ensemble (soft-voting): PatchInferenceEngine and patch_inference accept a list of models.
# Per-patch probabilities are averaged, then a single argmax/threshold is applied.
import tempfile, os, nibabel as nib
from monai.networks.nets import UNet
from monai.networks.layers import Norm

_ens_data = np.random.randn(32, 32, 32).astype(np.float32)
_ens_nii = nib.Nifti1Image(_ens_data, np.eye(4))

def _mk_ens_unet(out_channels=2):
    return UNet(spatial_dims=3, in_channels=1, out_channels=out_channels,
                channels=(16, 32), strides=(2,), num_res_units=1, norm=Norm.INSTANCE).eval()

with tempfile.TemporaryDirectory() as tmpdir:
    img_path = os.path.join(tmpdir, 'ens_img.nii.gz')
    nib.save(_ens_nii, img_path)

    _cfg = PatchConfig(patch_size=[32, 32, 32])
    _m1 = _mk_ens_unet()

    # A single-element list is stored as .models; .model aliases the first model (back-compat)
    _eng_list = PatchInferenceEngine([_m1], _cfg, apply_reorder=False)
    test_eq(len(_eng_list.models), 1)
    assert _eng_list.model is _m1

    # Ensembling a model with itself must equal single-model inference (identical probs -> same argmax)
    _pred_single = PatchInferenceEngine(_m1, _cfg, apply_reorder=False).predict(img_path)
    _pred_dup = PatchInferenceEngine([_m1, _m1], _cfg, apply_reorder=False).predict(img_path)
    assert torch.equal(_pred_single, _pred_dup), "Ensembling identical models should equal single-model output"

    # A list of two different models runs and returns a valid label mask (labels subset of {0, 1})
    _m2 = _mk_ens_unet()
    _eng_ens = PatchInferenceEngine([_m1, _m2], _cfg, apply_reorder=False)
    test_eq(len(_eng_ens.models), 2)
    _pred_ens = _eng_ens.predict(img_path)
    test_eq(_pred_ens.shape, _pred_single.shape)
    assert set(torch.unique(_pred_ens).tolist()).issubset({0, 1})

    # Ensemble + TTA runs and returns the same shape
    _pred_ens_tta = _eng_ens.predict(img_path, tta=True)
    test_eq(_pred_ens_tta.shape, _pred_single.shape)

    # patch_inference accepts a list of models (batch ensemble)
    _batch = patch_inference([_m1, _m2], _cfg, [img_path], apply_reorder=False, progress=False)
    test_eq(len(_batch), 1)
    test_eq(_batch[0].shape, _pred_single.shape)

    # Ensembling identical models via patch_inference equals the single-model result
    _batch_dup = patch_inference([_m1, _m1], _cfg, [img_path], apply_reorder=False, progress=False)
    assert torch.equal(_batch_dup[0], _pred_single)

    # Multi-class: keep_largest_component is binary-only, so it is skipped (with a warning) for >2 classes
    import warnings as _warnings
    _mc = _mk_ens_unet(out_channels=3)
    _mc_klc = PatchConfig(patch_size=[32, 32, 32], keep_largest_component=True)
    _mc_off = PatchConfig(patch_size=[32, 32, 32], keep_largest_component=False)
    _p_off = PatchInferenceEngine([_mc, _mc], _mc_off, apply_reorder=False).predict(img_path)
    with _warnings.catch_warnings(record=True) as _w:
        _warnings.simplefilter("always")
        _p_klc = PatchInferenceEngine([_mc, _mc], _mc_klc, apply_reorder=False).predict(img_path)
    assert torch.equal(_p_klc, _p_off), "keep_largest_component must be skipped for multi-class output"
    assert any("binary-only" in str(_wi.message) for _wi in _w), "expected a binary-only warning for multi-class"
    assert set(torch.unique(_p_klc).tolist()).issubset({0, 1, 2})

# An empty model list is rejected
test_fail(lambda: PatchInferenceEngine([], PatchConfig(patch_size=[32, 32, 32])))

print("Ensemble (soft-voting) tests passed!")

In [ ]:
#| hide
# A torch.compile'd model is unwrapped for inference, with one warning, and the
# caller's object is left alone. torch.compile needs a C++ toolchain and is slow in
# CI, so an nn.Module holding the inner module as `_orig_mod` stands in for
# OptimizedModule.
import warnings as _warnings
import torch.nn as _nn
from fastai.test_utils import synth_learner


class _OrigModWrapper(_nn.Module):
    def __init__(self, mod):
        super().__init__()
        self._orig_mod = mod


_ctest_model = _mk_ens_unet()
_ctest_cfg = PatchConfig(patch_size=[32, 32, 32], apply_reorder=False)

_wrapped = _OrigModWrapper(_ctest_model)
with _warnings.catch_warnings(record=True) as _w:
    _warnings.simplefilter("always")
    _eng = PatchInferenceEngine(_wrapped, _ctest_cfg)
assert _eng.model is _ctest_model
assert any('torch.compile' in str(_wi.message) for _wi in _w)
assert _wrapped._orig_mod is _ctest_model, "caller's wrapper must not be modified"

# a Learner argument is not mutated either
_lrn = synth_learner(n_trn=2, n_val=2)
_lrn.model = _OrigModWrapper(_ctest_model)
_eng = PatchInferenceEngine(_lrn, _ctest_cfg)
assert _eng.model is _ctest_model
test_eq(type(_lrn.model).__name__, '_OrigModWrapper')

# an uncompiled model produces no compile warning
with _warnings.catch_warnings(record=True) as _w:
    _warnings.simplefilter("always")
    PatchInferenceEngine(_ctest_model, _ctest_cfg)
assert not any('torch.compile' in str(_wi.message) for _wi in _w)

print("torch.compile unwrap tests passed!")

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()